# Vanguard A/B Test – Data Cleaning (team version)

**This is the only place where the data is cleaned.** All analysis notebooks only *read* the 3 CSV files saved here.

**Output files:**
- `cleaned_clients_demographics.csv` – demographics of the clients in the experiment
- `cleaned_clients_variation.csv` – test group per client (`Control` / `Test`)
- `cleaned_web_events.csv` – web events of the clients in the experiment (pt_1 + pt_2 combined)

**Team decisions made here:**
1. Only clients who are in the experiment (Control or Test) are kept in all 3 files.
2. Rare gender value `X` is merged into `U` (Unknown).
**Run this notebook first** (Kernel → Restart & Run All), then the analysis notebooks.

## 1. Import libraries

In [1]:
import pandas as pd

## 2. Load raw data
The raw files must be in the same folder as this notebook. Check that the file names match your files.

In [2]:
df_demo = pd.read_csv('01_original_data_files/df_final_demo.txt')
df_exp = pd.read_csv('01_original_data_files/df_final_experiment_clients.txt')
df_web_pt1 = pd.read_csv('01_original_data_files/df_final_web_data_pt_1.txt')
df_web_pt2 = pd.read_csv('01_original_data_files/df_final_web_data_pt_2.txt')

## 3. Clean the data

### 3.1 Demographics data
- Drop rows where `clnt_tenure_yr` is missing (these are the empty rows).
- Fill the missing `clnt_age` values with the **median** age.
- Merge the rare gender value `X` into `U` (Unknown).

In [3]:
df_demo = df_demo.dropna(subset=['clnt_tenure_yr'])
median_age = df_demo['clnt_age'].median()
df_demo['clnt_age'] = df_demo['clnt_age'].fillna(median_age)
df_demo['gendr'] = df_demo['gendr'].replace('X', 'U')

### 3.2 Experiment data
- Drop clients with no test group (`Variation` is missing). They were not part of the A/B test.

In [4]:
df_exp = df_exp.dropna(subset=['Variation'])

### 3.3 Web tracking data
- Combine the two web data files (pt_1 and pt_2) into one table.
- Convert `date_time` from text to datetime.
- Remove exact duplicate rows.

In [5]:
df_web = pd.concat([df_web_pt1, df_web_pt2], ignore_index=True)
df_web['date_time'] = pd.to_datetime(df_web['date_time'])
df_web = df_web.drop_duplicates()

### 3.4 Keep only clients who are in the experiment
So that everybody analyses **the same group of clients**, the demographics and web data are filtered to the clients that appear in the experiment file (Control or Test).

`.isin()` checks, for each row, whether its `client_id` is in the experiment list (True/False), and only the True rows are kept. The experiment file itself is also filtered to clients that have demographics.

In [6]:
df_exp = df_exp[df_exp['client_id'].isin(df_demo['client_id'])]
exp_ids = df_exp['client_id']

df_demo = df_demo[df_demo['client_id'].isin(exp_ids)]
df_web = df_web[df_web['client_id'].isin(exp_ids)]

## 4. Quick checks before saving

In [7]:
print('demographics:', df_demo.shape, '| unique clients:', df_demo['client_id'].nunique())
print('variation:   ', df_exp.shape, '| unique clients:', df_exp['client_id'].nunique())
print('web events:  ', df_web.shape, '| unique clients:', df_web['client_id'].nunique())
print()
print('missing values:', df_demo.isna().sum().sum(), df_exp.isna().sum().sum(), df_web.isna().sum().sum())
print('duplicate client_id in demographics:', df_demo['client_id'].duplicated().sum())
print('duplicate client_id in variation:   ', df_exp['client_id'].duplicated().sum())
print()
print(df_demo['gendr'].value_counts(dropna=False))
print(df_exp['Variation'].value_counts())

demographics: (50488, 9) | unique clients: 50488
variation:    (50488, 2) | unique clients: 50488
web events:   (317135, 5) | unique clients: 50488

missing values: 0 0 0
duplicate client_id in demographics: 0
duplicate client_id in variation:    0

gendr
U    17282
M    16947
F    16259
Name: count, dtype: int64
Variation
Test       26961
Control    23527
Name: count, dtype: int64


**Expected:** no missing values, no duplicate `client_id` in demographics or variation, only `F` / `M` / `U` in `gendr`, and only `Control` / `Test` in `Variation`.

## 5. Save cleaned datasets to CSV

In [9]:
# df_demo.to_csv('cleaned_clients_demographics.csv', index=False)
# df_exp.to_csv('cleaned_clients_variation.csv', index=False)
# df_web.to_csv('cleaned_web_events.csv', index=False)

print("All datasets successfully cleaned and saved to CSV files!")

All datasets successfully cleaned and saved to CSV files!


## 6. Data model (how the 3 tables connect)

All three tables are linked by `client_id`.

```
[ clients_demographics ]
  ├── client_id          (PK)  unique client ID
  ├── clnt_tenure_yr     tenure in years
  ├── clnt_tenure_mnth   tenure in months
  ├── clnt_age           age (missing values filled with median)
  ├── gendr              gender: F / M / U
  ├── num_accts          number of accounts
  ├── bal                total balance
  ├── calls_6_mnth       calls in the last 6 months
  └── logons_6_mnth      logins in the last 6 months
          │ 1:1
          ▼
[ clients_variation ]
  ├── client_id          (FK)
  └── Variation          'Control' or 'Test'
          │ 1:N  (one client has many web events)
          ▼
[ web_events ]
  ├── client_id          (FK)
  ├── visitor_id         visitor / device ID
  ├── visit_id           session visit ID
  ├── process_step       start / step_1 / step_2 / step_3 / confirm
  └── date_time          event timestamp
```